In [ ]:
import numpy as np
import pandas as pd
import marimo as mo

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import RidgeCV

from xgboost import XGBRegressor

# データの読み込み
train = pd.read_csv("kaggle_competitions/kaggle_student_test_scores_prediction/data/train.csv")
test = pd.read_csv("kaggle_competitions/kaggle_student_test_scores_prediction/data/test.csv")
sample_submission = pd.read_csv("kaggle_competitions/kaggle_student_test_scores_prediction/data/sample_submission.csv")

## 1. ライブラリのインポートとデータの読み込み

まずは分析に必要なライブラリを読み込み、学習データ・テストデータ・提出用サンプルファイルを読み込む。
その後、データの先頭を確認して全体像を把握する。

## 2. RidgeCVで使用するカラムと評価指標の準備

まずは1段目のモデルとして RidgeCV を学習させる。
そのために、数値変数とカテゴリ変数のカラムを定義し、評価指標として RMSE を計算する関数を用意する。

In [ ]:
# RidgeCVで使用する数値変数のカラムを指定
ridge_numeric_cols = [
    "age",
    "study_hours",
    "class_attendance",
    "sleep_hours",
]

# RidgeCVで使用するカテゴリ変数のカラムを指定
ridge_categorical_cols = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "facility_rating",
    "exam_difficulty",
]

# RMSEを計算する関数
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## 3. RidgeCVによる1段目モデルの学習

まずは1段目のモデルとして RidgeCV を学習させる。
ここでは、元の特徴量から予測値を作り、その予測値を後段のモデルに渡すための準備を行う。
このように、あるモデルの予測値を別のモデルの入力として使う考え方は、stacking の基本的な発想に近い。

In [ ]:
# RidgeCVで探索するalphaの候補を設定
ridgecv_alphas = np.logspace(-3, 3, 25)

# RidgeCV用の前処理を定義
# 数値変数は標準化し、カテゴリ変数はOneHotEncoderで数値化する
ridgecv_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ridge_numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ridge_categorical_cols),
    ]
)

# 前処理 + RidgeCVをまとめたパイプラインを作成
ridgecv_pipeline = Pipeline([
    ("preprocessor", ridgecv_preprocessor),
    ("model", RidgeCV(alphas=ridgecv_alphas, cv=5, scoring="neg_root_mean_squared_error")),
])

# 目的変数を分離し、説明変数を作成
X_ridgecv_base = train.drop(columns=["exam_score"]).copy()
X_ridgecv_base_test = test.copy()
y_ridgecv_base = train["exam_score"].copy()

# 5-foldの交差検証を設定
kf_ridgecv = KFold(n_splits=5, shuffle=True, random_state=42)

# OOF予測とテスト予測を保存する配列を用意
ridgecv_oof_pred = np.zeros(len(X_ridgecv_base))
ridgecv_test_pred = np.zeros(len(X_ridgecv_base_test))

# 各foldで選ばれたbest alphaを保存するリスト
ridgecv_best_alpha_list = []

for ridgecv_fold, (ridgecv_train_idx, ridgecv_valid_idx) in enumerate(
    kf_ridgecv.split(X_ridgecv_base, y_ridgecv_base), start=1
):
    # foldごとに学習用データと検証用データを分割
    X_ridgecv_train_fold = X_ridgecv_base.iloc[ridgecv_train_idx].copy()
    X_ridgecv_valid_fold = X_ridgecv_base.iloc[ridgecv_valid_idx].copy()
    y_ridgecv_train_fold = y_ridgecv_base.iloc[ridgecv_train_idx].copy()
    y_ridgecv_valid_fold = y_ridgecv_base.iloc[ridgecv_valid_idx].copy()

    # 学習
    ridgecv_pipeline.fit(X_ridgecv_train_fold, y_ridgecv_train_fold)

    # 検証データに対する予測
    ridgecv_valid_pred_fold = ridgecv_pipeline.predict(X_ridgecv_valid_fold)
    ridgecv_oof_pred[ridgecv_valid_idx] = ridgecv_valid_pred_fold

    # テストデータに対する予測をfoldごとに平均化
    ridgecv_test_pred_fold = ridgecv_pipeline.predict(X_ridgecv_base_test)
    ridgecv_test_pred += ridgecv_test_pred_fold / kf_ridgecv.n_splits

    # foldごとのRMSEとbest alphaを記録
    ridgecv_fold_rmse = rmse(y_ridgecv_valid_fold, ridgecv_valid_pred_fold)
    ridgecv_best_alpha = ridgecv_pipeline.named_steps["model"].alpha_
    ridgecv_best_alpha_list.append(ridgecv_best_alpha)

    print(
        f"RidgeCV Fold {ridgecv_fold}: "
        f"RMSE = {ridgecv_fold_rmse:.5f}, alpha = {ridgecv_best_alpha:.6f}"
    )

# 全foldのOOF予測を使ってCV RMSEを計算
ridgecv_cv_rmse = rmse(y_ridgecv_base, ridgecv_oof_pred)
print(f"\nRidgeCV CV RMSE: {ridgecv_cv_rmse:.5f}")
print(f"Fold best alphas: {ridgecv_best_alpha_list}")
print(f"Mean alpha: {np.mean(ridgecv_best_alpha_list):.6f}")

RidgeCV Fold 1: RMSE = 8.88649, alpha = 10.000000
RidgeCV Fold 2: RMSE = 8.89207, alpha = 17.782794
RidgeCV Fold 3: RMSE = 8.88514, alpha = 10.000000
RidgeCV Fold 4: RMSE = 8.89738, alpha = 10.000000
RidgeCV Fold 5: RMSE = 8.91300, alpha = 10.000000

RidgeCV CV RMSE: 8.89482
Fold best alphas: [np.float64(10.0), np.float64(17.78279410038923), np.float64(10.0), np.float64(10.0), np.float64(10.0)]
Mean alpha: 11.556559


## 4. 特徴量エンジニアリング関数の定義

次に、XGBoostで使用する追加特徴量を作成する。
今回は `study_hours` の変換特徴量と、一部カテゴリ変数の順序特徴量を追加する。
また、`study_hours` のビン分割は train と test で境界がずれないように、両者をまとめて処理する。

In [ ]:
def add_engineered_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    # trainとtestで同じ基準の特徴量を作るため、一度結合してから処理する
    all_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    # study_hours の2乗特徴量
    all_df["study_hours_squared"] = all_df["study_hours"] ** 2

    # study_hours の対数変換
    all_df["log_study_hours"] = np.log1p(all_df["study_hours"].clip(lower=0))

    # study_hours の平方根変換
    all_df["sqrt_study_hours"] = np.sqrt(all_df["study_hours"].clip(lower=0))

    # study_hours を5分割したビン番号
    all_df["study_bin_num"] = pd.cut(all_df["study_hours"], bins=5, labels=False)
    all_df["study_bin_num"] = all_df["study_bin_num"].fillna(0).astype(int)

    # 順序情報を持つカテゴリ変数を数値化
    ordinal_maps = {
        "sleep_quality": {"poor": 0, "average": 1, "good": 2},
        "facility_rating": {"low": 0, "medium": 1, "high": 2},
        "exam_difficulty": {"easy": 0, "moderate": 1, "hard": 2},
    }

    all_df["sleep_quality_ord"] = all_df["sleep_quality"].map(ordinal_maps["sleep_quality"])
    all_df["facility_rating_ord"] = all_df["facility_rating"].map(ordinal_maps["facility_rating"])
    all_df["exam_difficulty_ord"] = all_df["exam_difficulty"].map(ordinal_maps["exam_difficulty"])

    # 元のtrain/testの長さで分割し直す
    train_fe = all_df.iloc[: len(train_df)].copy()
    test_fe = all_df.iloc[len(train_df):].copy()

    return train_fe, test_fe

## 5. RidgeCVの予測値を新しい特徴量として追加

次に、RidgeCV で得られた予測値を `ridge_pred` として説明変数に追加する。
この `ridge_pred` は、1段目モデルが捉えた線形的な傾向を要約した特徴量とみなすことができる。
そのうえで、XGBoost に元の特徴量と追加特徴量の両方を入力し、より柔軟な非線形関係を学習させる。

In [ ]:
# 目的変数を分離し、説明変数を作成
X_xgb_ridge_base = train.drop(columns=["exam_score"]).copy()
X_xgb_ridge_base_test = test.copy()
y_xgb_ridge = train["exam_score"].copy()

# RidgeCVの予測値を新しい特徴量として追加
X_xgb_ridge_base["ridge_pred"] = ridgecv_oof_pred
X_xgb_ridge_base_test["ridge_pred"] = ridgecv_test_pred

# 特徴量エンジニアリングを適用
X_xgb_ridge_fe, X_test_xgb_ridge_fe = add_engineered_features(
    X_xgb_ridge_base,
    X_xgb_ridge_base_test,
)

## 6. RidgeCV予測を加えたXGBoostモデルの学習

ここでは、元の特徴量に加えて `ridge_pred` と追加の特徴量エンジニアリング結果を使い、XGBoost を学習させる。
つまり、1段目の RidgeCV が捉えた傾向を2段目の XGBoost に引き継ぐことで、単体モデルより高い予測性能を目指す構成になっている。

In [ ]:
# XGBoostで使用する数値変数のカラムを指定
xgb_ridge_numeric_cols = [
    "age",
    "study_hours",
    "class_attendance",
    "sleep_hours",
    "ridge_pred",
    "study_hours_squared",
    "log_study_hours",
    "sqrt_study_hours",
    "study_bin_num",
    "sleep_quality_ord",
    "facility_rating_ord",
    "exam_difficulty_ord",
]

# XGBoostで使用するカテゴリ変数のカラムを指定
xgb_ridge_categorical_cols = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "facility_rating",
    "exam_difficulty",
]

# 前処理の定義
# 数値変数はそのまま使用し、カテゴリ変数はOneHotEncoderで数値化する
xgb_ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", xgb_ridge_numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), xgb_ridge_categorical_cols),
    ]
)

# XGBoost回帰モデルの定義
xgb_ridge_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

# 5-foldの交差検証を設定
kf_xgb_ridge = KFold(n_splits=5, shuffle=True, random_state=42)

# OOF予測とテスト予測を保存する配列を用意
xgb_ridge_oof_pred = np.zeros(len(X_xgb_ridge_fe))
xgb_ridge_test_pred = np.zeros(len(X_test_xgb_ridge_fe))

for xgb_ridge_fold, (xgb_ridge_train_idx, xgb_ridge_valid_idx) in enumerate(
    kf_xgb_ridge.split(X_xgb_ridge_fe, y_xgb_ridge), start=1
):
    # foldごとに学習用データと検証用データを分割
    X_xgb_ridge_train_fold = X_xgb_ridge_fe.iloc[xgb_ridge_train_idx].copy()
    X_xgb_ridge_valid_fold = X_xgb_ridge_fe.iloc[xgb_ridge_valid_idx].copy()
    y_xgb_ridge_train_fold = y_xgb_ridge.iloc[xgb_ridge_train_idx].copy()
    y_xgb_ridge_valid_fold = y_xgb_ridge.iloc[xgb_ridge_valid_idx].copy()

    # 前処理 + モデルをまとめたパイプラインを作成
    xgb_ridge_pipeline = Pipeline([
        ("preprocessor", xgb_ridge_preprocessor),
        ("model", xgb_ridge_model),
    ])

    # 学習
    xgb_ridge_pipeline.fit(X_xgb_ridge_train_fold, y_xgb_ridge_train_fold)

    # 検証データに対する予測
    xgb_ridge_valid_pred_fold = xgb_ridge_pipeline.predict(X_xgb_ridge_valid_fold)
    xgb_ridge_oof_pred[xgb_ridge_valid_idx] = xgb_ridge_valid_pred_fold

    # テストデータに対する予測をfoldごとに平均化
    xgb_ridge_test_pred_fold = xgb_ridge_pipeline.predict(X_test_xgb_ridge_fe)
    xgb_ridge_test_pred += xgb_ridge_test_pred_fold / kf_xgb_ridge.n_splits

    # foldごとのRMSEを計算
    xgb_ridge_fold_rmse = rmse(y_xgb_ridge_valid_fold, xgb_ridge_valid_pred_fold)
    print(f"XGB+RidgePred Fold {xgb_ridge_fold}: RMSE = {xgb_ridge_fold_rmse:.5f}")

# 全foldのOOF予測を使ってCV RMSEを計算
xgb_ridge_cv_rmse = rmse(y_xgb_ridge, xgb_ridge_oof_pred)
print(f"\nXGB+RidgePred CV RMSE: {xgb_ridge_cv_rmse:.5f}")

XGB+RidgePred Fold 1: RMSE = 8.72035
XGB+RidgePred Fold 2: RMSE = 8.72831
XGB+RidgePred Fold 3: RMSE = 8.71939
XGB+RidgePred Fold 4: RMSE = 8.73901
XGB+RidgePred Fold 5: RMSE = 8.75374

XGB+RidgePred CV RMSE: 8.73217


## 7. 提出ファイルの作成

交差検証で平均したテスト予測を提出形式に整え、CSVファイルとして保存する。

In [ ]:
# 提出ファイルの作成
submission = sample_submission.copy()
submission["exam_score"] = xgb_ridge_test_pred

# CSVとして保存
submission.to_csv(
    "kaggle_competitions/kaggle_student_test_scores_prediction/output/submission_study_fe_xgb_ridge_pred.csv",
    index=False,
)

print("\nSaved: submission_study_fe_xgb_ridge_pred.csv")


Saved: submission_study_fe_xgb_ridge_pred.csv
